# 03 Feature Engineering - Mini Trade Promo Optimizer

Esta fase construye predictores explicables y disponibles antes de la tanda para futuros modelos de `uplift_real` y `roi`. No entrena modelos, no ajusta transformadores y no recomienda promociones.

## 1. Objetivo

Construir features que permitan simular como cambian la respuesta y el retorno esperado al modificar descuento y duracion, preservando contexto de SKU, cadena, baseline, elasticidad, calendario y mecanica secundaria.

## 2. Contexto de negocio

El uplift es porcentual, pero el impacto depende del volumen base durante la tanda. El descuento puede elevar respuesta y reducir ROI; la duracion aumenta exposicion sin una relacion monotona simple. Por ello se crean escalas, curvaturas e interacciones con significado comercial, sin atribuir causalidad.

## 3. Trazabilidad del codigo

In [ ]:
import pandas as pd

traceability = pd.DataFrame([
    ["Pipeline fase 03", "run_feature_engineering", "mini_tpo.pipeline", "todos los outputs", "Ejecucion unica"],
    ["Alineamiento", "validate_input_alignment", "mini_tpo.feature_engineering", "feature_engineering_validation.csv", "Join 1:1 y anti-leakage"],
    ["Features core", "build_engineered_core", "mini_tpo.feature_engineering", "model_features_engineered_core.parquet", "Transformaciones deterministicas"],
    ["Features opcionales", "build_engineered_optional", "mini_tpo.feature_engineering", "model_features_engineered_optional.parquet", "Sensibilidad y cold start"],
    ["Catalogo", "build_feature_catalog", "mini_tpo.feature_engineering", "feature_engineering_catalog.csv", "Justificacion y gobierno"],
    ["Compatibilidad", "build_optimizer_compatibility", "mini_tpo.feature_engineering", "feature_optimizer_compatibility.csv", "Recalculo por escenario"],
    ["Redundancia", "engineered_feature_redundancy", "mini_tpo.feature_engineering", "engineered_feature_redundancy.csv", "Model risk"],
], columns=["resultado", "funcion", "modulo", "output", "proposito"])
display(traceability)

## 4. Configuracion y ejecucion unica

In [ ]:
from pathlib import Path
import sys
import json
import importlib
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "configs" / "project_config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from mini_tpo.data_loading import load_config
import mini_tpo.feature_sets as feature_sets_module
import mini_tpo.feature_engineering as feature_engineering_module
import mini_tpo.pipeline as pipeline_module

# Recarga explicita para sesiones interactivas que ya tenian imports del proyecto.
feature_sets_module = importlib.reload(feature_sets_module)
feature_engineering_module = importlib.reload(feature_engineering_module)
pipeline_module = importlib.reload(pipeline_module)

cfg = load_config(ROOT / "configs" / "project_config.yaml")
FIGURES_DIR = ROOT / cfg["project"]["feature_engineering_figures_dir"]
TABLES_DIR = ROOT / cfg["project"]["tables_dir"]
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

# Unica llamada al pipeline de feature engineering en el notebook.
result = pipeline_module.run_feature_engineering()
paths = result["paths"]

## 5. Datos de entrada

In [ ]:
safe = pd.read_parquet(ROOT / cfg["project"]["safe_features"])
targets = pd.read_parquet(ROOT / cfg["project"]["safe_targets"])
index = pd.read_parquet(ROOT / cfg["project"]["safe_index"])
core = pd.read_parquet(paths["engineered_core"])
optional = pd.read_parquet(paths["engineered_optional"])

inputs = pd.DataFrame([
    ["model_features_safe", *safe.shape, safe.row_id.is_unique],
    ["model_targets", *targets.shape, targets.row_id.is_unique],
    ["model_index", *index.shape, index.row_id.is_unique],
], columns=["artifact", "filas", "columnas", "row_id_unico"])
display(inputs)

Los tres artifacts de fase 02 son entradas de solo lectura. La fecha se recupera desde el indice; targets y flags de evaluacion permanecen separados de las features.

## 6. Validacion de alineamiento

In [ ]:
validation = result["validation_summary"]
display(validation)
assert validation.passed.all()
joined_ids = safe[["row_id"]].merge(targets[["row_id"]], on="row_id", validate="one_to_one").merge(index[["row_id"]], on="row_id", validate="one_to_one")
print("Filas alineadas 1:1:", len(joined_ids))

**Decision:** cualquier diferencia de filas, duplicado o join no 1:1 detiene el pipeline. Esto protege la correspondencia entre contexto, fecha y targets sin depender de posiciones implícitas.

## 7. Principios anti-leakage

In [ ]:
from mini_tpo.constants import POST_PROMO_COLUMNS, AUDIT_COLUMNS, TARGET_UPLIFT, TARGET_ROI

forbidden = set(POST_PROMO_COLUMNS) | set(AUDIT_COLUMNS) | {TARGET_UPLIFT, TARGET_ROI}
anti_leakage = pd.DataFrame([
    ["Targets ausentes", not {TARGET_UPLIFT, TARGET_ROI}.intersection(core.columns)],
    ["Postpromocion ausente", not set(POST_PROMO_COLUMNS).intersection(core.columns)],
    ["Auditoria ausente", not set(AUDIT_COLUMNS).intersection(core.columns)],
    ["Fecha directa ausente", "fecha_inicio_tanda" not in core.columns],
    ["Promedios target ausentes", not any("mean_uplift" in c or "mean_roi" in c for c in core.columns)],
], columns=["control", "cumple"])
display(anti_leakage)

No se ajustan encoders, escaladores, imputadores ni selectores. El piso de uplift queda para evaluacion. `uplift_real` no puede predecir ROI; un futuro `uplift_predicho` debera generarse out-of-fold.

## 8. Feature sets originales

In [ ]:
display(pd.DataFrame({"feature_original_segura": safe.columns.drop("row_id")}))
display(pd.DataFrame({"feature_core": feature_sets_module.FEATURES_ENGINEERED_CORE}))

Las siete variables originales seguras se conservan. Las derivadas no sustituyen automaticamente originales porque lineales y arboles pueden beneficiarse de representaciones distintas.

## 9. Duracion en semanas

In [ ]:
display(core[["duracion_dias", "duracion_semanas"]].drop_duplicates().sort_values("duracion_dias"))
assert np.allclose(core.duracion_semanas, core.duracion_dias / 7)

**Formula:** `duracion_semanas = duracion_dias / 7`. **Fundamento:** alinea la duracion con `volumen_base_sem`. **Negocio:** facilita leer la exposicion en la misma unidad del baseline. **Riesgo:** es redundante para modelos lineales sin regularizacion. **Decision:** core y documentada.

## 10. Volumen base durante la tanda

In [ ]:
display(core[["volumen_base_sem", "duracion_dias", "volumen_base_tanda"]].describe())
assert np.allclose(core.volumen_base_tanda, core.volumen_base_sem * core.duracion_dias / 7)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(core.volumen_base_tanda, bins=45, color="#4E79A7", edgecolor="white")
ax.set(title="Volumen base esperado durante la tanda", xlabel="Unidades base de la tanda", ylabel="Promociones")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "volumen_base_tanda_distribution.png", dpi=150); plt.show()

**Formula:** `volumen_base_tanda = volumen_base_sem * duracion_dias / 7`. **Negocio:** permite convertir luego `uplift_predicho` en unidades: `volumen_incremental_esperado = volumen_base_tanda * uplift_predicho`. Esta ultima variable no se calcula porque todavia no existe prediccion. **Decision:** feature core y recalculable por duracion.

## 11. Magnitud de elasticidad

In [ ]:
display(core[["elasticidad_estimada", "elasticidad_abs"]].describe())
assert np.allclose(core.elasticidad_abs, core.elasticidad_estimada.abs())

**Formula:** `elasticidad_abs = abs(elasticidad_estimada)`. **Fundamento:** expresa sensibilidad en magnitud; el signo original tambien se conserva. **Negocio:** una magnitud mayor implica respuesta estimada mas sensible al precio. **Limitacion:** es una estimacion interna, no causal.

## 12. Interaccion descuento x elasticidad

In [ ]:
analysis = core.merge(targets[["row_id", "uplift_real", "roi"]], on="row_id", validate="one_to_one")
fig, ax = plt.subplots(figsize=(8, 5))
hb = ax.hexbin(analysis.descuento_x_elasticidad, analysis.uplift_real, gridsize=30, mincnt=1, cmap="viridis")
ax.set(title="Presion descuento-elasticidad versus uplift", xlabel="factor_descuento x elasticidad_abs", ylabel="Uplift observado")
fig.colorbar(hb, ax=ax, label="Observaciones")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "discount_elasticity_vs_uplift.png", dpi=150); plt.show()

**Formula:** `descuento_x_elasticidad = factor_descuento * elasticidad_abs`. **Negocio:** la misma profundidad puede tener respuestas distintas segun sensibilidad. **Riesgo:** la relacion descriptiva con uplift no prueba efecto causal ni justifica seleccion por si sola. **Decision:** interaccion core, especialmente interpretable en modelos lineales.

## 13. Interaccion descuento x duracion

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
hb = ax.hexbin(analysis.descuento_x_duracion, analysis.uplift_real, gridsize=25, mincnt=1, cmap="plasma")
ax.set(title="Intensidad descuento-duracion versus uplift", xlabel="factor_descuento x duracion_dias", ylabel="Uplift observado")
fig.colorbar(hb, ax=ax, label="Observaciones")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "discount_duration_vs_uplift.png", dpi=150); plt.show()

**Formula:** `descuento_x_duracion = factor_descuento * duracion_dias`. Resume profundidad y exposicion, pero no representa inversion real. Cambia con ambas palancas y debe recalcularse para cada candidato.

## 14. No linealidad de descuento y duracion

In [ ]:
display(core[["factor_descuento", "factor_descuento_sq", "duracion_dias", "duracion_dias_sq"]].describe())
assert np.allclose(core.factor_descuento_sq, core.factor_descuento ** 2)
assert np.allclose(core.duracion_dias_sq, core.duracion_dias ** 2)

Los cuadrados permiten curvatura, saturacion o rendimientos decrecientes en modelos lineales regularizados. No implican que la relacion verdadera sea cuadratica y pueden ser redundantes para arboles; se conservan para comparacion futura dentro de folds.

## 15. Transformaciones logaritmicas

In [ ]:
log_comparison = pd.DataFrame([
    ["volumen_base_sem", core.volumen_base_sem.skew(), core.log1p_volumen_base_sem.skew()],
    ["volumen_base_tanda", core.volumen_base_tanda.skew(), core.log1p_volumen_base_tanda.skew()],
], columns=["variable", "asimetria_original", "asimetria_log1p"])
display(log_comparison)
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, values, title in [
    (axes[0,0], core.volumen_base_sem, "Volumen base semanal"),
    (axes[0,1], core.log1p_volumen_base_sem, "log1p volumen base semanal"),
    (axes[1,0], core.volumen_base_tanda, "Volumen base de tanda"),
    (axes[1,1], core.log1p_volumen_base_tanda, "log1p volumen base de tanda"),
]:
    ax.hist(values, bins=40, color="#59A14F", edgecolor="white"); ax.set_title(title); ax.set_ylabel("Promociones")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "log_transformations.png", dpi=150); plt.show()

`log1p` reduce asimetria y peso de SKUs grandes sin eliminar las escalas originales. Es valida porque los volumenes son no negativos. Su utilidad predictiva se evaluara temporalmente, no solo por distribucion.

## 16. Features temporales

In [ ]:
temporal_cols = feature_sets_module.FEATURES_ENGINEERED_TEMPORAL
display(core[temporal_cols].describe())
temporal_plot = core[["row_id", "mes_sin", "mes_cos", "semana_anio_sin", "semana_anio_cos"]].merge(index[["row_id", "fecha_inicio_tanda"]], on="row_id", validate="one_to_one").groupby("fecha_inicio_tanda").first().sort_index()
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
temporal_plot[["mes_sin", "mes_cos"]].plot(ax=axes[0]); axes[0].set_title("Codificacion ciclica mensual"); axes[0].set_ylabel("Valor ciclico")
temporal_plot[["semana_anio_sin", "semana_anio_cos"]].plot(ax=axes[1]); axes[1].set_title("Codificacion ciclica semanal"); axes[1].set_ylabel("Valor ciclico")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "temporal_cyclical_features.png", dpi=150); plt.show()

Mes, trimestre y semana son conocidos al disenar la tanda. Seno y coseno preservan cercania entre fin e inicio del ciclo. La fecha original permanece en `model_index`; `dias_desde_inicio_dataset` queda opcional por riesgo de extrapolacion de tendencia.

## 17. Features categoricas

In [ ]:
categorical_profile = pd.DataFrame([
    [column, core[column].nunique(dropna=False), str(core[column].dtype)]
    for column in feature_sets_module.FEATURES_CATEGORICAL
], columns=["feature", "categorias", "dtype"])
display(categorical_profile)
linear_template = feature_engineering_module.build_preprocessor("linear")
tree_template = feature_engineering_module.build_preprocessor("tree")
print("Template lineal sin fit:", linear_template)
print("Template arbol sin fit:", tree_template)

SKU, cadena y flag secundario conservan su naturaleza categorica. No se realiza one-hot global ni target encoding. En modelado, `OneHotEncoder(handle_unknown='ignore')` y el escalamiento se ajustaran dentro de cada fold temporal.

## 18. Features opcionales

In [ ]:
display(optional[["row_id", *feature_sets_module.FEATURES_ENGINEERED_OPTIONAL]].head())
display(pd.DataFrame({"feature_opcional": feature_sets_module.FEATURES_ENGINEERED_OPTIONAL}))
print("Combinaciones sku_cadena:", optional.sku_cadena.nunique())

Precio y marca son deterministas respecto a SKU en este historico; pueden ayudar en ROI, arboles o cold start, pero no pertenecen al core. `sku_cadena` captura heterogeneidad local con riesgo de sobreajuste. Missing secundario y tendencia temporal requieren validacion de sensibilidad.

## 19. Features historicas no incorporadas

In [ ]:
catalog = pd.read_csv(paths["feature_engineering_catalog"])
display(catalog.loc[catalog.estado.eq("future fold-aware")])

No se materializan medias de uplift, ROI ni resultados promocionales. Si se evaluan despues, deben usar observaciones estrictamente anteriores, `shift`, cortes por fecha y transformacion fold-aware. Promociones de una misma fecha no pueden informarse entre si.

## 20. Correlacion y redundancia

In [ ]:
correlation = pd.read_csv(paths["engineered_feature_correlation"], index_col=0)
redundancy = pd.read_csv(paths["engineered_feature_redundancy"])
target_association = pd.read_csv(paths["engineered_feature_target_association"])
display(redundancy)
display(target_association)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(correlation.values, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
ax.set_xticks(range(len(correlation.columns)), correlation.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(correlation.index)), correlation.index, fontsize=7)
ax.set_title("Correlacion Spearman de features numericas")
fig.colorbar(im, ax=ax, label="Spearman")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "engineered_feature_correlation.png", dpi=150); plt.show()

Las correlaciones altas son esperables entre variables y sus transformaciones. No se eliminan automaticamente: modelos lineales requeriran regularizacion y ablation; arboles pueden ignorar representaciones redundantes. Correlacion baja tampoco implica inutilidad si existe interaccion, no linealidad o valor de escala comercial. Las asociaciones con uplift y ROI son diagnosticas: no se usan para seleccion automatica ni alteran los artifacts de features.

## 21. Compatibilidad con el optimizador

In [ ]:
compatibility = pd.read_csv(paths["feature_optimizer_compatibility"])
display(compatibility)
display(compatibility.loc[compatibility.cambia_con_descuento | compatibility.cambia_con_duracion])

Las variables de contexto permanecen fijas para un SKU, cadena y fecha. Para cada candidato, el optimizador futuro debe recalcular descuento, duracion, baseline de tanda, interacciones, cuadrados y logaritmo dependiente de duracion. Esta tabla es un contrato de simulacion, no una recomendacion.

## 22. Feature sets para uplift y ROI

In [ ]:
sets = pd.DataFrame({
    "uplift": pd.Series(feature_sets_module.FEATURES_FOR_UPLIFT),
    "roi": pd.Series(feature_sets_module.FEATURES_FOR_ROI),
})
display(sets)

Descuento, elasticidad e interacciones son centrales para uplift. Baseline de tanda ayuda especialmente a representar escala para ROI. Precio queda opcional. `uplift_real` nunca entra al modelo de ROI; un uplift predicho solo podria usarse out-of-fold durante entrenamiento.

## 23. Catalogo de features

In [ ]:
display(catalog)
assert catalog.interpretacion_tecnica.str.strip().ne("").all()
assert catalog.interpretacion_negocio.str.strip().ne("").all()

El catalogo documenta formula, fuente, disponibilidad, leakage, uso por target, comportamiento en simulacion y estado. Es la fuente de trazabilidad de esta fase.

## 24. Validaciones finales

In [ ]:
display(validation.groupby(["dataset", "passed"]).size().to_frame("controles"))
assert core.shape == (2048, 24)
assert optional.shape == (2048, 29)
assert core.row_id.tolist() == safe.row_id.tolist()
assert not core.isna().any().any() and not optional.isna().any().any()
print("Todos los controles pasaron:", validation.passed.all())

## 25. Artifacts generados

In [ ]:
artifact_view = pd.DataFrame({
    "artifact": paths.keys(),
    "ruta_relativa": [path.relative_to(ROOT) for path in paths.values()],
    "existe": [path.exists() for path in paths.values()],
})
display(artifact_view)
manifest = json.loads(paths["feature_engineering_manifest"].read_text(encoding="utf-8"))
print("Manifest version:", manifest["version"], "| filas:", manifest["row_count"])

## 26. Limitaciones y preparacion para modelado

- Las relaciones mostradas con uplift y ROI son descriptivas, no causales ni criterios unicos de seleccion.
- Precio, marca, `sku_cadena`, missing y tendencia temporal requieren validacion temporal de sensibilidad.
- No se materializan historicos con targets ni uplift predicho.
- No se ajustaron encoders, escaladores, imputadores, selectores ni modelos.
- La siguiente fase debe definir folds temporales, ajustar preprocesamiento dentro de cada fold, comparar un baseline minimo y medir error porcentual y en unidades por SKU y cadena.
- El optimizador solo comenzara tras validar capacidad predictiva, estabilidad y soporte historico.